# 02 · Wallet Features

Author: Chris Kim — 2026-05, ChainSense · UCSD DS3

The gold layer holds one feature vector per wallet, built from the silver tables. This notebook does two things: inspect what landed in data/gold (the v1 → v5 evolution) and document the standardization that turns the raw features into the model-ready matrix. The feature-building queries live in the pipeline scripts — here we only look at the result.


## 1. What's in data/gold

Each version added features; v5 is the one that goes to the model. Below: the files present, how the column set grew v1 → v5, a peek at the raw values, and the selector lookup that backs the *_share columns.


In [ ]:
import duckdb
con = duckdb.connect()

# the gold feature parquets (v1 → v5 plus the standardized / model variants)
con.sql("SELECT file FROM glob('../data/gold/*.parquet') ORDER BY file")


In [ ]:
# how the feature set (labels) grew across versions
for v in ["v1", "v2", "v3", "v4", "v5_unstandardized"]:
    cols = con.execute(f"DESCRIBE SELECT * FROM '../data/gold/wallet_features_{v}.parquet'").df()["column_name"].tolist()
    print(f"{v}  ({len(cols)} cols): {', '.join(cols)}")


In [ ]:
# a peek at the final raw (unstandardized) feature values
con.sql("SELECT * FROM '../data/gold/wallet_features_v5_unstandardized.parquet' LIMIT 5")


In [ ]:
# selector lookup behind the *_share columns
con.sql("SELECT * FROM '../data/gold/top_20_selectors.parquet'")


## 2. Standardization

Every feature is z-scored (StandardScaler, population std) before clustering — HDBSCAN is distance-based, so unscaled heavy-tailed columns would dominate. Transform applied per column:

| Feature | Scaling | Note |
|---|---|---|
| tx_count | z-score | log-transform first would've been better; heavy-tailed |
| account_age_days | z-score | |
| active_days | z-score | |
| failed_tx_ratio | z-score | already [0,1], so z-scoring is optional but harmless for distance methods |
| unique_peers | z-score | heavy-tailed; log first would've helped |
| total_volume_eth_out | z-score | very heavy-tailed; log strongly advised |
| in_volume_eth | z-score | same; log advised |
| median_tx_interval_sec | z-score | |
| tx_interval_std_sec | z-score | |
| same_block_tx_count | z-score | |
| counterparty_concentration_top3 | z-score | already [0,1] |
| contract_call_ratio | z-score | already [0,1] |
| selector_entropy | z-score | |
| active_hour_entropy | z-score | |
| median_priority_fee_gwei | z-score | heavy-tailed |
| selector_share_top* (14 cols) | z-score | already [0,1] fractions; mathematically fine, interpretively odd |
| distinct_tokens | z-score | |
| token_churn_rate | z-score | NaN→0 imputed after scaling (known issue — collapses zero-token wallets into the mean) |


## 3. Unstandardized → standardized → model

Three short steps from wallet_features_v5_unstandardized.parquet:

1. drop rows with any NA — the clustering input must be complete.
2. z-score every feature → wallet_features_v5_standardized.parquet.
3. drop the address key → wallet_features_v5_model.parquet, the pure numeric matrix HDBSCAN consumes.


In [ ]:
SRC = "../data/gold/wallet_features_v5_unstandardized.parquet"
feat = [c for c in con.execute(f"DESCRIBE SELECT * FROM '{SRC}'").df()["column_name"] if c != "address"]

# 1. drop rows with any NA
clean = f"SELECT * FROM '{SRC}' WHERE " + " AND ".join(f"{c} IS NOT NULL" for c in feat)

# 2. z-score every feature: (x - mean) / std   (token_churn_rate NaN->0 after scaling)
z = ", ".join(f"(({c} - avg({c}) OVER ()) / stddev_pop({c}) OVER ()) AS {c}" for c in feat)
con.execute(f"""
    COPY (SELECT address, {z} FROM ({clean}))
    TO '../data/gold/wallet_features_v5_standardized.parquet' (FORMAT PARQUET)
""")

# 3. drop the address key -> model-ready matrix for HDBSCAN
con.execute("""
    COPY (SELECT * EXCLUDE (address)
          FROM '../data/gold/wallet_features_v5_standardized.parquet')
    TO '../data/gold/wallet_features_v5_model.parquet' (FORMAT PARQUET)
""")
